# 10年定着予測 - 探索的データ分析レポート v4（希望勤務地マッチ度・昇給タイミング）

**背景**: `20_`〜`24_`で入社時メモ構造化（ブロックE）がPublicで唯一確実に改善したブロックだった一方、
自己学習実施月数（G）・早期離職シグナル深掘り（H）・人物所見キーワード（I）はいずれもEに追加すると
Publicで悪化するという結果が3回連続で続いた（`data/output/submit_result_report.md`セクション23-31参照）。
この教訓を踏まえ、v4では**Eのような「客観的な事実・条件のマッチング」系の情報源**を優先的に探索する。

1. 希望勤務地と実際の配属地のマッチ度（推奨・最優先）
2. 給与の昇給タイミング・パターン
3. （副産物）初任給の勤務地内偏差
4. 転居許容と実際の転居発生の一致度（新規探索）
5. 上司・同僚からのフィードバックのキーワード探索（新規探索）
6. 未使用の生データ列点検（採用経路・性別・初期勤務地・初期役割）

In [ ]:
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")  # Colab想定。ローカル実行時は適宜変更
INPUT_DIR = PROJECT_ROOT / "data" / "input"

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")

y = train_persona[TARGET_COL]
print(f"Train Persona: {train_persona.shape}, Train Monthly: {train_monthly.shape}")
print(f"定着率: {y.mean():.4f}")

## 1. 希望勤務地と実際の配属地のマッチ度（最重要発見）

`入社時メモ`から抽出した「希望勤務地」（ブロックEで既に抽出済み）と、既存の構造化列「初期勤務地」
（実際の配属地）を比較し、**希望通りに配属されたかどうか**を新たに検証する。両者は個別には
既にモデルに投入されているが、その"一致・不一致"という関係性自体は未検証だった。

In [ ]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


def extract_desired_location(s):
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


ws_section = train_persona["入社時メモ"].apply(extract_workstyle_section)
desired_location = ws_section.apply(extract_desired_location)
actual_location = train_persona["初期勤務地"]

valid = desired_location.notna()
match = desired_location == actual_location

print(f"希望勤務地を抽出できた行数: {valid.sum()} / {len(train_persona)}")
print(f"一致率: {match[valid].mean():.4f}")
print()
print("マッチ有無別の定着率:")
print(y[valid].groupby(match[valid]).agg(["mean", "count"]))

ct = pd.crosstab(match[valid], y[valid])
chi2, p, _, _ = chi2_contingency(ct)
print(f"\nカイ二乗検定: chi2={chi2:.3f}, p={p:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
rates = y[valid].groupby(match[valid]).mean()
ax.bar(["不一致", "一致"], [rates[False], rates[True]], color=["salmon", "steelblue"])
ax.axhline(y.mean(), color="gray", linestyle="--", label=f"全体平均({y.mean():.3f})")
ax.set_title("希望勤務地マッチ度 別 定着率")
ax.set_ylabel("定着率")
ax.legend()
plt.tight_layout()
plt.show()

### 考察（1）

希望勤務地に配属された社員の定着率は**62.1%**、そうでない社員は**36.8%**と、**25.3%ptもの差**が
確認できた（p=1.4×10⁻²⁶）。これは本プロジェクト全体を通じて確認された効果量の中で最大級である。
希望勤務地・初期勤務地はそれぞれ単独では既にモデルに投入済みだが、**両者の一致・不一致という
関係性自体は決定木モデルが自動的には効率よく学習できない可能性があり、明示的な特徴量として
価値が高い**と考えられる。

## 2. 給与の昇給タイミング・パターン

月例給与の推移（slope/diff等）は既に特徴量化されているが、**最初の昇給が何ヶ月目に発生したか**という
タイミング構造は未分析だった。

In [ ]:
def first_raise_month(g):
    g = g.sort_values("経過月数")
    salaries = g["月例給与_円"].values
    months = g["経過月数"].values
    base = salaries[0]
    for i in range(1, len(salaries)):
        if salaries[i] > base:
            return months[i]
    return np.nan


raise_month = train_monthly.groupby(ID_COL, group_keys=False).apply(first_raise_month, include_groups=False)
raise_month = raise_month.reindex(train_persona[ID_COL]).reset_index(drop=True)

print(f"昇給が発生した社員数: {raise_month.notna().sum()} / {len(raise_month)}")
print(raise_month.describe())

In [ ]:
early_flag = raise_month <= 6
valid_raise = raise_month.notna()

print("早期昇給フラグ(<=6ヶ月) 別 定着率:")
print(y[valid_raise].groupby(early_flag[valid_raise]).agg(["mean", "count"]))

ct = pd.crosstab(early_flag[valid_raise], y[valid_raise])
chi2, p, _, _ = chi2_contingency(ct)
print(f"\nカイ二乗検定: chi2={chi2:.3f}, p={p:.2e}")

no_raise = raise_month.isna()
print(f"\n昇給なし群(n={no_raise.sum()})の定着率: {y[no_raise].mean():.3f}（参考、サンプル数が少なく解釈注意）")

fig, ax = plt.subplots(figsize=(6, 5))
rates = y[valid_raise].groupby(early_flag[valid_raise]).mean()
ax.bar(["通常(7ヶ月目以降)", "早期(6ヶ月以内)"], [rates[False], rates[True]], color=["salmon", "steelblue"])
ax.axhline(y.mean(), color="gray", linestyle="--", label=f"全体平均({y.mean():.3f})")
ax.set_title("初回昇給タイミング 別 定着率")
ax.set_ylabel("定着率")
ax.legend()
plt.tight_layout()
plt.show()

### 考察（2）

入社6ヶ月以内に最初の昇給があった社員の定着率は**68.8%**、それ以降（多くは12ヶ月目、定期昇給と
思われる）の社員は**54.6%**と、**14.2%ptの差**が確認できた（p=6.7×10⁻⁷）。早期昇給は
高評価・高ポテンシャル人材への特別な処遇である可能性があり、新規特徴量として有望。
昇給なし群（n=21）は少数すぎるため参考値にとどめる。

## 3. 初任給の勤務地内偏差（副産物）

分析1の調査中に、初任給を勤務地別平均からの偏差として見ると緩やかな関連があることに気づいたため、
併せて記録する（等級内偏差・区分内偏差は既存特徴量だが、勤務地内偏差は未計算だった）。

In [ ]:
loc_mean_salary = train_persona.groupby("初期勤務地")["初任給_円"].transform("mean")
salary_dev = train_persona["初任給_円"] - loc_mean_salary

print(f"相関係数: {salary_dev.corr(y):.4f}")
bins = pd.qcut(salary_dev, q=5)
print(y.groupby(bins, observed=True).agg(["mean", "count"]))

### 考察（3）

相関係数は0.130とやや弱いが、最上位分位（勤務地平均より初任給が高いグループ）では定着率69.9%と
明確に高く、非線形な関係が示唆される。分析1・2ほど強くはないが、追加の候補として記録する。

## 4. 転居許容と実際の転居発生の一致度（新規探索、null）

「転居を伴う異動を許容しない」と回答した社員が、実際に勤務地変更（転居）を経験した場合に
定着率へ影響するかを確認する。

In [ ]:
NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


reloc_pref = ws_section.apply(classify_reloc)
actual_reloc = (train_monthly.groupby(ID_COL)["勤務地"].nunique() > 1).reindex(train_persona[ID_COL]).values

print(f"実際に転居(勤務地変更)した社員数: {actual_reloc.sum()} / {len(actual_reloc)}")

mismatch = (reloc_pref == False) & actual_reloc
valid_reloc = reloc_pref.notna()
print(f"「許容せず」なのに実際に転居した社員数: {mismatch[valid_reloc].sum()}")
print(y[valid_reloc].groupby(mismatch[valid_reloc]).agg(["mean", "count"]))

### 考察（4）

実際に転居（勤務地変更）を経験した社員は全体でわずか60名（2.2%）しかおらず、「許容しないと回答した
のに実際に転居させられた」ケースは20名にとどまる。定着率の差もほぼない（55.0% vs 56.3%）。
**サンプル数不足のため有意な結論を出せず、不採用**とする。

## 5. 上司・同僚からのフィードバックのキーワード探索（新規探索、null）

`入社時メモ`の「人物所見」（ブロックI）と同じ手法を、より長文の「上司からのフィードバック」
「同僚からのフィードバック」にも適用できないか探索した。

In [ ]:
candidates = {
    "自発性キーワード": ["自ら", "自発的", "自分から"],
    "促され系キーワード": ["促され", "促す", "働きかけを要", "働きかけを受け"],
    "振り返り学習キーワード": ["振り返り", "学び"],
    "相談キーワード": ["相談"],
    "市場価値志向キーワード": ["市場価値", "社外でも通用", "処遇"],
    "安定継続志向キーワード": ["安定", "継続して経験", "同じ環境"],
}

for col in ["上司からのフィードバック", "同僚からのフィードバック"]:
    print(f"=== {col} ===")
    text = train_persona[col].fillna("")
    for name, kws in candidates.items():
        flag = text.apply(lambda t: any(k in t for k in kws))
        if flag.sum() < 20 or (~flag).sum() < 20:
            continue
        ct = pd.crosstab(flag, y)
        chi2, p, _, _ = chi2_contingency(ct)
        print(f"{name}: n={flag.sum()}, chi2 p={p:.4f}")
    print()

### 考察（5）

いずれのキーワードも統計的に有意ではなかった（全てp>0.15）。「人物所見」（1〜2文の定型的な短文）と
異なり、フィードバック文はより長文・多様な言い回しで書かれており、単純なキーワード一致では
シグナルを検出できないと考えられる。**この方向は不採用**とする。

## 6. 未使用の生データ列点検

これまでのノートブックで一度も明示的に触れていなかった列（採用経路・性別・初期勤務地・初期役割）を
点検した。

In [ ]:
for col in ["採用経路", "性別", "初期勤務地", "初期役割"]:
    ct = pd.crosstab(train_persona[col], y)
    chi2, p, _, _ = chi2_contingency(ct)
    print(f"=== {col} (chi2 p={p:.2e}) ===")
    print(train_persona.groupby(col)[TARGET_COL].agg(["mean", "count"]))
    print()

### 考察（6）

いずれも統計的に有意な差がある列だったが（採用経路・初期役割はp<0.0001）、`drop_cols`に
含まれておらず、**実は全て既にCatBoostへ生のカテゴリ変数として投入済み**であることが判明した。
新規性はないが、モデルがこれらの強い情報を利用できていること自体は確認できた。

## 7. 総合考察と次のアクション

| # | 分析 | 判定 | 効果量 |
|---|---|---|---|
| 1 | 希望勤務地マッチ度 | **最有望** | 差25.3%pt, p=1.4×10⁻²⁶ |
| 2 | 早期昇給フラグ | **有望** | 差14.2%pt, p=6.7×10⁻⁷ |
| 3 | 初任給の勤務地内偏差 | 弱い有望 | 相関0.130 |
| 4 | 転居許容×実際の転居ミスマッチ | 不採用(null) | サンプル数不足(n=20) |
| 5 | 上司・同僚フィードバックのキーワード | 不採用(null) | 全て非有意 |
| 6 | 未使用列点検 | 新規性なし | 既にモデルに投入済み |

### 次のアクション

`20_`〜`24_`で確立した教訓（E_memoに新ブロックを追加すると3回連続でPublicが悪化した）を踏まえ、
分析1・2（・3）で見つかった新ブロックは**E_memoに追加せず、まず18_のベースライン（Eなし）に対して
単体で検証する**。特に分析1（希望勤務地マッチ度）は本プロジェクト全体で最大級の効果量であり、
優先的に検証する。